In [9]:
line = line = "2 458207137259 eni-089831853bd9d5eb7 41.168.10.139 172.31.35.206 22462 23 6 1 40 1782509589 1782509617 REJECT OK"

for index, line in enumerate(line.split()):
    print(f"Index: {index}: {line}")



Index: 0: 2
Index: 1: 458207137259
Index: 2: eni-089831853bd9d5eb7
Index: 3: 41.168.10.139
Index: 4: 172.31.35.206
Index: 5: 22462
Index: 6: 23
Index: 7: 6
Index: 8: 1
Index: 9: 40
Index: 10: 1782509589
Index: 11: 1782509617
Index: 12: REJECT
Index: 13: OK


In [10]:
def parse_flow(line):
    parts = line.split()
    return {
        "srcaddr": parts[3],
        "dstaddr": parts[4],
        "dstport": parts[6],
        "action": parts[12],
    }

In [11]:
from collections import Counter

attacker_ips = Counter()

with open(r"C:\Users\Krzychu\Documents\Python\Cloud\data\logs.log", "r") as file:
    for line in file:
        parsed = parse_flow(line)
        if parsed["action"] == "REJECT":
            attacker_ips[parsed["srcaddr"]] += 1


print(attacker_ips.most_common(10))

[('41.168.10.139', 15), ('210.245.120.117', 11), ('104.194.10.16', 8), ('204.76.203.51', 6), ('85.217.140.35', 5), ('79.124.56.146', 4), ('45.87.249.146', 3), ('85.217.140.25', 3), ('85.217.140.13', 3), ('85.217.140.3', 3)]


In [12]:
import pandas as pd

df_ip = pd.DataFrame(attacker_ips.most_common(10), columns=["IP", "Attempts"])
df_ip



,IP,Attempts
0,41.168.10.139,15
1,210.245.120.117,11
2,104.194.10.16,8
3,204.76.203.51,6
4,85.217.140.35,5
5,79.124.56.146,4
6,45.87.249.146,3
7,85.217.140.25,3
8,85.217.140.13,3
9,85.217.140.3,3


In [13]:
PORT_NAMES = {
    "22": "SSH", "23": "Telnet", "80": "HTTP", "443": "HTTPS",
    "3389": "RDP", "445": "SMB", "8080": "HTTP-alt", "8443": "HTTPS-alt",
    "21": "FTP", "10000": "Webmin", "2003": "Carbon/Graphite"
}

top_attacked_ports = Counter()

with open(r"C:\Users\Krzychu\Documents\Python\Cloud\data\logs.log", "r") as file:
    for line in file:
        parsed = parse_flow(line)
        if parsed["action"] == "REJECT":
            top_attacked_ports[parsed["dstport"]] += 1


print(top_attacked_ports.most_common(10))

df_ports = pd.DataFrame(top_attacked_ports.most_common(10), columns=["Port", "Attempts"])
df_ports["Service"] = df_ports["Port"].map(PORT_NAMES)
df_ports

[('23', 36), ('80', 6), ('10000', 3), ('8443', 2), ('54530', 2), ('8840', 2), ('22011', 2), ('9470', 2), ('45234', 2), ('2003', 2)]


,Port,Attempts,Service
0,23,36,Telnet
1,80,6,HTTP
2,10000,3,Webmin
3,8443,2,HTTPS-alt
4,54530,2,NaN
5,8840,2,NaN
6,22011,2,NaN
7,9470,2,NaN
8,45234,2,NaN
9,2003,2,Carbon/Graphite


In [14]:
total = 0
rejects = 0
accepts = 0
unique_attackers = set()

with open(r"C:\Users\Krzychu\Documents\Python\Cloud\data\logs.log", "r") as file:
    for line in file:
        parsed = parse_flow(line)
        if parsed["action"] == "REJECT":
            rejects += 1
            unique_attackers.add(parsed["srcaddr"])
        if parsed["action"] == "ACCEPT":
            accepts += 1

print(f"Number of rejects: {rejects}")
print(f"Number of accepts: {accepts}")
print(f"Total number of actions: {rejects + accepts}")    
print(f"Unique attacker's IPs: {len(unique_attackers)}")  
print(f"Blocked: {rejects / (rejects + accepts) *100:.1f}%")  

Number of rejects: 217
Number of accepts: 187
Total number of actions: 404
Unique attacker's IPs: 125
Blocked: 53.7%


In [ ]:
ip_ports = {}

with open(r"C:\Users\Krzychu\Documents\Python\Cloud\data\logs.log", "r") as file:
    for line in file:
        parsed = parse_flow(line)
        if parsed["action"] == "REJECT":
            ip = parsed["srcaddr"]
            port = parsed["dstport"]
            if ip not in ip_ports:
                ip_ports[ip] = set()
            ip_ports[ip].add(port)


for ip, ports in ip_ports.items():
    if len(ports) >= 3:
        print(f"IP: {ip} -> {len(ports)} {ports} ports")





AttributeError: 'dict' object has no attribute 'item'